# Fit one fixed configuration on synthetic folds

This minimal companion constructs 12 synthetic observations, fits a fresh pipeline
once per fold, and inspects held-out predictions. It does not load football data
or choose a pilot model. Use the existing `misc314_py314` kernel.

The [guide](../docs/analytics/training.md) adds stored feature selection,
classification and custom adapters; the [reference](../docs/analytics/training_reference.md)
lists every parameter and output field.

## 1. Prepare aligned data and folds

In ordinary use, dataset assembly and the split library provide these objects.
Here the original integer positions and scoring subsets are written explicitly.

In [1]:
import numpy as np
import pandas as pd
from xdiyo_analytics.datasets import ModelDataset
from xdiyo_analytics.splits import Fold, SplitPlan
from xdiyo_analytics.training import EstimatorAdapter, TrainingRunner, fit_predict

positions = np.arange(12)
X = pd.DataFrame({
    'recent': [2., 3., np.nan, 4., 2., 5., 3., 6., 4., 7., 5., 8.],
    'venue': positions % 2,
    'trend': positions / 10,
})
y = pd.DataFrame({'quantity': 3. + positions / 2 + positions % 3})
metadata = pd.DataFrame({
    'competition_id': 17, 'season_id': 2024,
    'event_id': pd.Series([2**63 + 101 + int(i) for i in positions], dtype='uint64[pyarrow]'),
    'kickoff_at': pd.date_range('2024-01-01', periods=12, freq='2D', tz='UTC'),
})
keys = ('competition_id', 'season_id', 'event_id')
dataset = ModelDataset(X, y, metadata, 'match', keys, keys, 'total',
                       {'features': {'kind': 'synthetic'}, 'label': {'kind': 'synthetic'}})
plan = SplitPlan([
    Fold(np.arange(6), np.arange(6, 9), np.array([7, 8]), {'stage': 'first'}),
    Fold(np.arange(9), np.arange(9, 12), np.array([], dtype=int), {'stage': 'second'}),
], n_rows=12, row_order=positions)


## 2. Fit fresh preprocessing and a model

The factory makes a new imputer, scaler and Ridge estimator for each fold. Their
fitted state uses training rows only. The runner returns every test prediction.

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

def model_factory():
    return EstimatorAdapter(make_pipeline(
        SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=1.0),
    ))

result = TrainingRunner(model_factory).run(dataset, plan)
result.prediction_frame()


quantity
fold_id row_position           
0       6              7.377203
        7              8.537813
        8              8.869938
1       9              9.319894
        10             9.475376
        11            10.631076

## 3. Inspect the scoring subset

Fold zero scores two of its three test rows; fold one has an empty scoring subset.
All six test predictions remain available in `result`. No metric is calculated.

In [3]:
result.prediction_frame(scored_only=True)

quantity
fold_id row_position          
0       7             8.537813
        8             8.869938

## 4. Inspect retained scope and fitted objects

Each fold result holds its fitted adapter, positions, columns, targets and metadata.
For example, `result.folds[0].model.estimator` is the first fitted pipeline.

In [4]:
pd.DataFrame([
    {'fold': fold.fold_id, 'train_rows': len(fold.train_positions),
     'test_rows': len(fold.test_positions), 'score_rows': len(fold.score_positions),
     'features': fold.feature_columns, 'targets': fold.target_columns}
    for fold in result.folds
])

,fold,train_rows,test_rows,score_rows,features,targets
0,0,6,3,2,"(recent, venue, trend)","(quantity,)"
1,1,9,3,0,"(recent, venue, trend)","(quantity,)"


## Next steps

The guide shows how to supply fixed column subsets or consume an existing
training-only feature selection. Search, nested evaluation, scheduled refitting
and post-training reporters remain later layers. Histories, ratings and
information cutoffs stay upstream; this runner cannot repair leaking features.

[Verification coverage](../docs/analytics/training_documentation_checklist.md) ·
[Verification evidence](../docs/analytics/training_check.json)